In [1]:
# Open count stores and run Scarf analyses.
import scarf
# Keep routine logs and progress bars out of the results.
scarf.configure_output(level="WARNING", progress=False)

# Download the prepared example store.
dataset = scarf.cytebase.connect("scarf_docs").download_dataset(
    "tenx_5K_pbmc_rnaseq",
    destination="scarf_datasets",
    zarr=True,
)
# Open the count store for this analysis.
ds = scarf.DataStore(f"{dataset}/data.zarr", nthreads=4)
# Open the saved analysis used to check reuse.
baseline_run = ds.pipeline.open(label="docs_default")
# Keep the cells used by the saved analysis.
cell_selection = baseline_run["analysis_cell_selection"]
# Keep the variable genes used by the saved analysis.
hvg_ref = baseline_run["highly_variable_features"]
# Inspect the opened store's cells and features.
ds

DataStore has 3948 (5025) cells with 1 assays: RNA
   Cell metadata:
            'I', 'ids', 'names', 'RNA_G2M_score', 'RNA_S_score', 
            'RNA_UMAP1', 'RNA_UMAP2', 'RNA_cell_cycle_phase', 'RNA_clusters', 'RNA_doublet_score', 
            'RNA_leiden_cluster', 'RNA_nCounts', 'RNA_nFeatures', 'RNA_paris_cluster', 'RNA_percentMito', 
            'RNA_percentRibo'
   RNA assay has 33538 features and following metadata:
            'I', 'ids', 'names', 'dropOuts', 'feature_type', 
            'genome', 'nCells'

In [2]:
# Reopen the saved normalized expression.
normalized = baseline_run["normalized"]
# Reopen the saved PCA result.
pca = baseline_run["pca"]
# Keep the saved nearest-neighbor index.
ann = baseline_run["ann_index"]
# Keep the saved 11-neighbor result.
neighbors_k11 = baseline_run["neighbors"]
# Keep the baseline connectivity graph.
graph_k11 = baseline_run["connectivity_map"]
# Inspect the saved chain from normalization to connectivity.
{
    "normalization": normalized,
    "PCA": pca,
    "neighbor index": ann,
    "neighbors": neighbors_k11,
    "graph": graph_k11,
}

{'normalization': ArtifactRef(assay='RNA', kind='normalized', artifact_id='e1d501833065...'),
 'PCA': ArtifactRef(assay='RNA', kind='reduction', artifact_id='1f6b522d2aed...'),
 'neighbor index': ArtifactRef(assay='RNA', kind='ann_index', artifact_id='5019e2243468...'),
 'neighbors': ArtifactRef(assay='RNA', kind='neighbors', artifact_id='fa09e9659d83...'),
 'graph': ArtifactRef(assay='RNA', kind='connectivity_map', artifact_id='c52919743a99...')}

In [3]:
# Find the exact saved graph from its neighbor input.
[reopened_graph] = ds.list_artifacts(
    from_assay="RNA",
    kind="connectivity_map",
    operation="build_connectivity_map",
    inputs={"neighbors": neighbors_k11},
    complete_only=True,
)
# Check that the catalog returned the baseline graph.
assert reopened_graph == graph_k11

# Inspect the saved graph's operation, parameters, and inputs.
status = ds.inspect_artifact(reopened_graph)
# Inspect the graph operation and the exact inputs it records.
{
    "operation": status.operation,
    "parameters": status.parameters,
    "inputs": status.inputs,
    "complete": status.complete,
}

{'operation': 'build_connectivity_map',
 'parameters': {'local_connectivity': 1.0, 'bandwidth': 1.5},
 'inputs': {'neighbors': {'type': 'artifact',
   'scope': 'assay',
   'kind': 'neighbors',
   'artifact_id': 'fa09e9659d83521bdd28483b7a0e038dd33e64262acaa0c5402627a9fbf15b57',
   'assay': 'RNA'}},
 'complete': True}

In [4]:
# Query the existing index with 15 neighbors per cell.
neighbors_k15 = ds.query_neighbors(ann, k=15)
# Build connectivity for the new neighbor count.
graph_k15 = ds.build_connectivity_map(neighbors_k15)

# Check which results were reused and which changed.
{
    "normalization reused": ds.run_normalization(cell_selection, hvg_ref) == normalized,
    "PCA reused": ds.run_pca(normalized, dims=15) == pca,
    "ANN index reused": ds.build_ann_index(pca) == ann,
    "neighbors recomputed": neighbors_k15 != neighbors_k11,
    "graph recomputed": graph_k15 != graph_k11,
}

{'normalization reused': True,
 'PCA reused': True,
 'ANN index reused': True,
 'neighbors recomputed': True,
 'graph recomputed': True}

In [5]:
# Fit a separate reduction with 20 principal components.
pca_dims20 = ds.run_pca(normalized, dims=20)
# Build a search index from the 20-component reduction.
ann_dims20 = ds.build_ann_index(pca_dims20)
# Find 11 neighbors using the new reduction.
neighbors_dims20 = ds.query_neighbors(ann_dims20, k=11)
# Build the connectivity graph for this reduction.
graph_dims20 = ds.build_connectivity_map(neighbors_dims20)

# Check which results were reused and which changed.
{
    "PCA recomputed": pca_dims20 != pca,
    "ANN index recomputed": ann_dims20 != ann,
    "neighbors recomputed": neighbors_dims20 != neighbors_k11,
    "graph recomputed": graph_dims20 != graph_k11,
}

{'PCA recomputed': True,
 'ANN index recomputed': True,
 'neighbors recomputed': True,
 'graph recomputed': True}

In [6]:
# Trace the shared inputs and differences between all three graphs.
lineage = ds.lineage(
    {
        "k11 graph": graph_k11,
        "k15 graph": graph_k15,
        "dims20 graph": graph_dims20,
    }
)
# Display the shared inputs and diverging analysis branches.
lineage

```mermaid
flowchart LR
    artifact0["RNA / metadata_snapshot | snapshot_run_metadata | 44d7dbfccbf0"]
    artifact1["datastore / cell_selection | snapshot_pipeline_input_selection | a5efbc1dac58"]
    artifact2["datastore / metadata_snapshot | snapshot_run_metadata | 916d464e738c"]
    artifact3["datastore / cell_selection | filter_pipeline_cells | e6e222f7a568"]
    artifact4["RNA / feature_summary | summarize_rna_features | 23338f376ef3"]
    artifact5["RNA / feature_selection | select_hvgs | cfd9446de944"]
    artifact6["RNA / normalized | run_normalization | e1d501833065"]
    artifact7["RNA / feature_scaling | calculate_feature_scaling | 3ba8ba9e7586"]
    artifact8["RNA / reduction | run_pca | 1f6b522d2aed"]
    artifact9["RNA / ann_index | build_ann_index | 5019e2243468"]
    artifact10["RNA / neighbors | query_neighbors | f792f3c3f594"]
    artifact11["RNA / connectivity_map | build_connectivity_map | db378b108419 | outputs: k15 graph"]
    artifact12["RNA / neighbors | query_neighbors | fa09e9659d83"]
    artifact13["RNA / connectivity_map | build_connectivity_map | c52919743a99 | outputs: k11 graph | status: stale (revision 1, current 2)"]
    artifact14["RNA / reduction | run_pca | b7c25d77d288"]
    artifact15["RNA / ann_index | build_ann_index | 3bc80322c7dd"]
    artifact16["RNA / neighbors | query_neighbors | 6a460a1f2742"]
    artifact17["RNA / connectivity_map | build_connectivity_map | a867b302c3ca | outputs: dims20 graph"]
    artifact8 -->|"coordinates"| artifact10
    artifact9 -->|"ann_index"| artifact10
    artifact10 -->|"neighbors"| artifact11
    artifact8 -->|"coordinates"| artifact12
    artifact9 -->|"ann_index"| artifact12
    artifact12 -->|"neighbors"| artifact13
    artifact3 -->|"pca_cell_selection"| artifact14
    artifact6 -->|"normalized"| artifact14
    artifact7 -->|"feature_scaling"| artifact14
    artifact14 -->|"coordinates"| artifact15
    artifact14 -->|"coordinates"| artifact16
    artifact15 -->|"ann_index"| artifact16
    artifact16 -->|"neighbors"| artifact17
    artifact1 -->|"input_cell_selection"| artifact3
    artifact2 -->|"cell_snapshot"| artifact3
    artifact3 -->|"cell_selection"| artifact4
    artifact0 -->|"feature_snapshot"| artifact5
    artifact4 -->|"feature_summary"| artifact5
    artifact3 -->|"cell_selection"| artifact6
    artifact5 -->|"feature_selection"| artifact6
    artifact6 -->|"normalized"| artifact7
    artifact3 -->|"pca_cell_selection"| artifact8
    artifact6 -->|"normalized"| artifact8
    artifact7 -->|"feature_scaling"| artifact8
    artifact8 -->|"coordinates"| artifact9
```

### Artifact details

#### RNA / metadata_snapshot / 44d7dbfccbf0
- Status: `complete`
- Path: `RNA/artifacts/metadata_snapshot/44d7dbfccbf005aa74c628e6883fa2a7524341b46046128bca8363b2fb9a1506`
- Operation: `snapshot_run_metadata`
- Parameters: `assay="RNA"; axis="feature"; ordered_columns=["names"]`
- Other inputs: `column_fingerprints={"names":"25b8d03a2f9b2b01183cc9742301f3cf9cd4762c4fe962fe3ad5189013d8c857"}; ordered_row_ids_fingerprint="4548c9820ab9ed0afbc48cbb6503bdcdc725b57c65dbfe3c4da8bb480b7fc4b1"`

#### datastore / cell_selection / a5efbc1dac58
- Status: `complete`
- Path: `artifacts/cell_selection/a5efbc1dac581a5c21453fc7802fc6bc1c08fcc68dce5d5240d026b297812310`
- Operation: `snapshot_pipeline_input_selection`
- Parameters: `assay="RNA"`
- Execution options: `source_column="I"`
- Other inputs: `ordered_row_ids_fingerprint="f5f05615ffc8833f75752b75152ccd728702f42950f81e9c28402880a4dd2ae5"; values_fingerprint="94f668fa448559768ea628c836d7ed1b683636c280cd42e9a6c9b73ba0179a38"`

#### datastore / metadata_snapshot / 916d464e738c
- Status: `complete`
- Path: `artifacts/metadata_snapshot/916d464e738cdd19a41efdd59283c7b447e707daed5e8bf61b6147124f1fe0d9`
- Operation: `snapshot_run_metadata`
- Parameters: `assay=null; axis="cell"; ordered_columns=["names","RNA_nCounts","RNA_nFeatures","RNA_percentMito"]`
- Other inputs: `column_fingerprints={"RNA_nCounts":"a8a7520000e32d28bcf97a8977290bcc7185570098e1fe95739c74687b435843","RNA_nFeatures":"a3df08addee7271194b0ebdfac85ad92ec93f3801031e65316776453eb...; ordered_row_ids_fingerprint="f5f05615ffc8833f75752b75152ccd728702f42950f81e9c28402880a4dd2ae5"`

#### datastore / cell_selection / e6e222f7a568
- Status: `complete`
- Path: `artifacts/cell_selection/e6e222f7a568767cf2c117990d277e956340d9480aa603d3c9c934a993c77539`
- Operation: `filter_pipeline_cells`
- Parameters: `attrs=["RNA_nCounts","RNA_nFeatures","RNA_percentMito"]; enabled=true; highs=[15000,4000,15]; keepBounds=false; lows=[1000,500,0]; method="manual"`
- Execution options: `source_column="I"`
- Other inputs: `ordered_row_ids_fingerprint="f5f05615ffc8833f75752b75152ccd728702f42950f81e9c28402880a4dd2ae5"; values_fingerprint="f994b6b61f3cddbb8b890fca9fee8f9b5e02f530ccf7645f21bf49719a79c39d"`

#### RNA / feature_summary / 23338f376ef3
- Status: `complete`
- Path: `RNA/artifacts/feature_summary/23338f376ef39421e128bc5b5cf31883eb5bb51a68c1d9d077cabdf57d3b7ecc`
- Operation: `summarize_rna_features`
- Parameters: `normalization_method={"module":"scarf.assay","qualname":"norm_lib_size"}; size_factor=1000`
- Execution options: `nthreads=2`
- Other inputs: `dataset_fingerprint="cf9c200795fa3b1c7645f5ef830af6fd277d319cf56c7ef4bae6e2da9bba4b5a"`

#### RNA / feature_selection / cfd9446de944
- Status: `complete`
- Path: `RNA/artifacts/feature_selection/cfd9446de944c63db9b3710166075c1635521bd02780aaaa9a0aa5c2159868f4`
- Operation: `select_hvgs`
- Parameters: `bin_strategy="adaptive"; blacklist="^MT-|^RPS|^RPL|^MRPS|^MRPL|^CCN|^HLA-|^H2-|^HIST|^XIST$|^DDX3Y$|^USP9Y$|^EIF1AY$|^KDM5D$|^SRY$|^ZFY$|^UTY$|^TMSB4Y$|^NLGN4Y$"; blacklist_fingerprint="40cb83e4a3e9a81770a7ef6b7841595d91720fc1928e937503f6ca23c089175e"; keep_bounds=false; lowess_frac=0.1; max_cells=3928; max_mean={"special_float":"inf"}; max_var={"special_float":"inf"}; min_cells=20; min_mean={"special_float":"-inf"}; min_var={"special_float":"-inf"}; n_bins=200; ... 3 more`
- Execution options: `invalidate_cache=false; nthreads=2; plot_kwargs={}; show_plot=false`

#### RNA / normalized / e1d501833065
- Status: `complete`
- Path: `RNA/artifacts/normalized/e1d50183306506cb8aff200c27e6c0f5f9731a548b286a59d7a22ca17570b57b`
- Operation: `run_normalization`
- Parameters: `log_transform=true; normalization_method={"external_hook":true,"module":"scarf.assay","qualname":"norm_lib_size"}; renormalize_subset=true; size_factor=1000.0`
- Execution options: `invalidate_cache=false`
- Other inputs: `dataset_fingerprint="cf9c200795fa3b1c7645f5ef830af6fd277d319cf56c7ef4bae6e2da9bba4b5a"`

#### RNA / feature_scaling / 3ba8ba9e7586
- Status: `complete`
- Path: `RNA/artifacts/feature_scaling/3ba8ba9e7586dc9348ecbce6a77f633aebc26a0f7ec7b71e23f9652245eec733`
- Operation: `calculate_feature_scaling`
- Parameters: `enabled=true`
- Execution options: `batch_size=3948; invalidate_cache=false; local_cache="auto"`

#### RNA / reduction / 1f6b522d2aed
- Status: `complete`
- Path: `RNA/artifacts/reduction/1f6b522d2aedafa62c58d8f646ae8803d55a2462b0e23716d20355df8a7c5e3c`
- Operation: `run_pca`
- Parameters: `dims=15; feat_scaling=true`
- Execution options: `batch_size=3948; invalidate_cache=false; local_cache="auto"; show_elbow_plot=false`

#### RNA / ann_index / 5019e2243468
- Status: `complete`
- Path: `RNA/artifacts/ann_index/5019e2243468867af0c4191cd9192efbb4cb2eb41ea5dc40ef75be9ffbea95dd`
- Operation: `build_ann_index`
- Parameters: `ann_ef=50; ann_efc=50; ann_m=48; ann_metric="l2"; ann_parallel=false; parallel_threads=null; rand_state=4466`
- Execution options: `batch_size=3948; invalidate_cache=false`

#### RNA / neighbors / f792f3c3f594
- Status: `complete`
- Path: `RNA/artifacts/neighbors/f792f3c3f594db0960560a4857b16fdc8c26742e32b8f67423e2532df0f5e215`
- Operation: `query_neighbors`
- Parameters: `distance_metric="l2"; k=15`
- Execution options: `batch_size=3948; invalidate_cache=false; nthreads=1`

#### RNA / connectivity_map / db378b108419
- Status: `complete`
- Path: `RNA/artifacts/connectivity_map/db378b1084196bb425fe1c8b2015710d10d0941777edd7728919ad7f0802b975`
- Operation: `build_connectivity_map`
- Revision: `2`
- Outputs: `k15 graph`
- Parameters: `bandwidth=1.5; local_connectivity=1.0`
- Execution options: `invalidate_cache=false`

#### RNA / neighbors / fa09e9659d83
- Status: `complete`
- Path: `RNA/artifacts/neighbors/fa09e9659d83521bdd28483b7a0e038dd33e64262acaa0c5402627a9fbf15b57`
- Operation: `query_neighbors`
- Parameters: `distance_metric="l2"; k=11`
- Execution options: `batch_size=3948; invalidate_cache=false`

#### RNA / connectivity_map / c52919743a99
- Status: `stale`
- Path: `RNA/artifacts/connectivity_map/c52919743a99fe5aeb9ad72faf0ba9602888ac3e6b105004dfecda7753d2f10c`
- Operation: `build_connectivity_map`
- Revision: `1, current 2: Edge weights are fitted with each cell itself at distance zero, as umap-learn does, so a cell's k weights sum to bandwidth * log2(k + 1) instead of bandwidth * log2(k) + 1`
- Outputs: `k11 graph`
- Parameters: `bandwidth=1.5; local_connectivity=1.0`
- Execution options: `invalidate_cache=false`

#### RNA / reduction / b7c25d77d288
- Status: `complete`
- Path: `RNA/artifacts/reduction/b7c25d77d2881cbee5f98818078dae64cbafd9a2e17bff85e72499adc77fab83`
- Operation: `run_pca`
- Parameters: `dims=20; feat_scaling=true`
- Execution options: `batch_size=3948; invalidate_cache=false; local_cache="auto"; show_elbow_plot=false`

#### RNA / ann_index / 3bc80322c7dd
- Status: `complete`
- Path: `RNA/artifacts/ann_index/3bc80322c7dd0c443a16b4dd0ba4225c6f10e0f87dea46a31377b481fd70059e`
- Operation: `build_ann_index`
- Parameters: `ann_ef=50; ann_efc=50; ann_m=48; ann_metric="l2"; ann_parallel=false; parallel_threads=null; rand_state=4466`
- Execution options: `batch_size=3948; invalidate_cache=false; nthreads=1`

#### RNA / neighbors / 6a460a1f2742
- Status: `complete`
- Path: `RNA/artifacts/neighbors/6a460a1f27423be8e78a905f1cad6d27cc3f25d2c3db647725d5e76b08f05435`
- Operation: `query_neighbors`
- Parameters: `distance_metric="l2"; k=11`
- Execution options: `batch_size=3948; invalidate_cache=false; nthreads=1`

#### RNA / connectivity_map / a867b302c3ca
- Status: `complete`
- Path: `RNA/artifacts/connectivity_map/a867b302c3ca7318febb42828d9acf84d8a1e859e27bb5ae65c01faeddce83c4`
- Operation: `build_connectivity_map`
- Revision: `2`
- Outputs: `dims20 graph`
- Parameters: `bandwidth=1.5; local_connectivity=1.0`
- Execution options: `invalidate_cache=false`

In [7]:
# Export the displayed lineage report as Markdown.
lineage_markdown = lineage.to_markdown()
# Preview the opening lines of the exported report.
lineage_markdown.splitlines()[:12]

['```mermaid',
 'flowchart LR',
 '    artifact0["RNA / metadata_snapshot | snapshot_run_metadata | 44d7dbfccbf0"]',
 '    artifact1["datastore / cell_selection | snapshot_pipeline_input_selection | a5efbc1dac58"]',
 '    artifact2["datastore / metadata_snapshot | snapshot_run_metadata | 916d464e738c"]',
 '    artifact3["datastore / cell_selection | filter_pipeline_cells | e6e222f7a568"]',
 '    artifact4["RNA / feature_summary | summarize_rna_features | 23338f376ef3"]',
 '    artifact5["RNA / feature_selection | select_hvgs | cfd9446de944"]',
 '    artifact6["RNA / normalized | run_normalization | e1d501833065"]',
 '    artifact7["RNA / feature_scaling | calculate_feature_scaling | 3ba8ba9e7586"]',
 '    artifact8["RNA / reduction | run_pca | 1f6b522d2aed"]',
 '    artifact9["RNA / ann_index | build_ann_index | 5019e2243468"]']